# 🧪 話者分離の実験（pyannote Community-1）

**最終更新日時: 2026-10-07 14:13**

本体（`notebook.ipynb`）は **変更しません**。このノートブックは実験専用です。
要件: `docs/spec_話者分離実験.md` ／ 計画: `docs/roadmap_話者分離実験.md`

## 使い方
1. **ランタイムを GPU（T4）に変更する**（「ランタイム」→「ランタイムのタイプを変更」）
2. Colab の「シークレット」に `HF_TOKEN`（Hugging Face の Read トークン）を登録して、ノートブックからのアクセスをオンにする
3. 上から順にセルを実行する（Step 1 のインストールのあと、ランタイムが自動で再起動します）

## 守っていること
- `01_input` の音声は **コピーするだけ**（移動・削除しません）
- Gemini・Notion・スプレッドシートは使いません

## 進み具合
- [x] Step 1: 土台（インストール・認証・音声の用意）
- [ ] Step 2: 条件 A（人数のヒントなし）
- [ ] Step 3: 条件 B（人数を指定）と比較
- [ ] Step 4: 文字起こしと突き合わせ


## Step 1-1: インストール
`pyannote.audio` を入れます（初回は数分かかります）。入れ終わると、ランタイムが自動で再起動されます。


In [ ]:
# ライブラリのインストール
# ※ 初回実行時は数分かかります
import time
start = time.time()

# uv（高速パッケージマネージャ）を使って一括インストール
!pip install -q uv
!uv pip install "pyannote.audio>=4.0" huggingface_hub

elapsed = int(time.time() - start)
print(f'✅ ライブラリのインストール完了（{elapsed}秒）')

# 🔔 完了音で通知（ド・ミ・ソの上昇音）
from IPython.display import Javascript, display
display(Javascript("""
var ctx = new AudioContext();
[440, 550, 660].forEach((freq, i) => {
  var osc = ctx.createOscillator();
  var gain = ctx.createGain();
  osc.connect(gain); gain.connect(ctx.destination);
  osc.frequency.value = freq;
  gain.gain.setValueAtTime(0.3, ctx.currentTime + i * 0.15);
  gain.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + i * 0.15 + 0.3);
  osc.start(ctx.currentTime + i * 0.15);
  osc.stop(ctx.currentTime + i * 0.15 + 0.3);
});
"""))

print('🔄 ランタイムを自動で再起動します（数秒後に再接続してください）...')

# インストール後に自動でランタイムを再起動（手動再起動の代わり）
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


---
## Step 1-2: 🔑 設定と認証
Google Drive のマウントと、Hugging Face のトークン・Community-1 の利用条件の確認をします。
ポップアップが出たら「許可」を押してください。


In [ ]:
import os
import shutil
import subprocess

from google.colab import drive, userdata

# ========================================
# 1. 設定
# ========================================
BASE_DIR        = '/content/drive/MyDrive/MeetingTranscript'
INPUT_AUDIO_DIR = f'{BASE_DIR}/01_input'
EXPERIMENT_DIR  = f'{BASE_DIR}/04_temp/experiment_diarization'  # 実験の結果の保存先（Colabが切れても残る）
WORK_DIR        = '/content/work'  # 音声のコピー先（Colab側の一時領域）
DIARIZATION_MODEL = 'pyannote/speaker-diarization-community-1'

# 実験で使う人数（条件 B で指定する。今回の音声は男性2人）
NUM_SPEAKERS = 2

# ========================================
# 2. HF_TOKEN の読み込み
# ========================================
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as e:
    raise ValueError(
        '❌ Colab のシークレット HF_TOKEN が読み込めません。\n'
        '   左のカギアイコン → HF_TOKEN を登録し、「ノートブックからのアクセス」をオンにしてください。\n'
        f'   （詳細: {e}）'
    )
if not HF_TOKEN:
    raise ValueError('❌ HF_TOKEN が空です。Hugging Face の Read トークンを登録してください。')
print('✅ HF_TOKEN 読み込み完了')

# ========================================
# 3. GPU と ffmpeg の確認
# ========================================
import torch
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ GPU が使えません。「ランタイム」→「ランタイムのタイプを変更」で T4 GPU にしてください（CPU でも動きますが遅いです）')

missing_tools = [tool for tool in ('ffmpeg', 'ffprobe') if shutil.which(tool) is None]
if missing_tools:
    raise RuntimeError(
        f'❌ {missing_tools} が見つかりません。\n'
        f'   新しいセルで !apt-get install -y ffmpeg を実行してから、もう一度このセルを実行してください。'
    )
print('✅ ffmpeg / ffprobe 確認OK')

# ========================================
# 4. Google Drive マウント
# ========================================
drive.mount('/content/drive')
print('✅ Google Drive マウント完了')

# ========================================
# 5. Community-1 の利用条件に同意済みか確認
#    （モデルの設定ファイルを1つ取得してみる。同意していないと GatedRepoError になる）
# ========================================
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import EntryNotFoundError, GatedRepoError, HfHubHTTPError

try:
    hf_hub_download(DIARIZATION_MODEL, 'config.yaml', token=HF_TOKEN)
    print(f'✅ {DIARIZATION_MODEL} にアクセスできます')
except EntryNotFoundError:
    # 設定ファイルの名前が違うだけで、モデル自体にはアクセスできている
    print(f'✅ {DIARIZATION_MODEL} にアクセスできます（設定ファイル名の確認はスキップ）')
except GatedRepoError:
    raise RuntimeError(
        f'❌ {DIARIZATION_MODEL} の利用条件に同意していません。\n'
        f'   https://huggingface.co/{DIARIZATION_MODEL} を開いて同意してから、もう一度実行してください。'
    )
except HfHubHTTPError as e:
    raise RuntimeError(
        f'❌ Hugging Face へのアクセスに失敗しました（トークンが無効・期限切れの可能性があります）。\n'
        f'   詳細: {e}'
    )


---
## Step 1-3: 音声の用意
`01_input` の音声を **コピー**して、話者分離に使う形式（16kHz・モノラルの wav）に変換します。
元のファイルは動かしません。音声の長さも表示します。


In [ ]:
# 対応する拡張子（本体と同じ考え方。動画も音声として読める）
AUDIO_EXTENSIONS = ('.mp3', '.m4a', '.wav', '.flac', '.ogg', '.aac', '.mp4', '.mov', '.avi', '.mkv', '.webm')

# 1. 入力フォルダから音声を探す（ファイル名順で先頭の1つを使う）
if not os.path.isdir(INPUT_AUDIO_DIR):
    raise FileNotFoundError(f'❌ 入力フォルダが見つかりません: {INPUT_AUDIO_DIR}')

audio_files = sorted(
    name for name in os.listdir(INPUT_AUDIO_DIR)
    if name.lower().endswith(AUDIO_EXTENSIONS)
)
if not audio_files:
    raise FileNotFoundError(f'❌ {INPUT_AUDIO_DIR} に音声ファイルがありません')

source_name = audio_files[0]
print(f'🎧 使う音声: {source_name}（{len(audio_files)}件中の先頭）')

# 2. Colab 側にコピー（Drive の元ファイルは動かさない）
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
source_ext = os.path.splitext(source_name)[1].lower()
copied_path = f'{WORK_DIR}/source{source_ext}'  # ファイル名の空白や日本語を避けるため固定名にする
shutil.copy2(f'{INPUT_AUDIO_DIR}/{source_name}', copied_path)
print('✅ コピー完了')

# 3. 16kHz・モノラルの wav に変換（-vn: 動画の映像は捨てる）
AUDIO_PATH = f'{WORK_DIR}/audio.wav'
result = subprocess.run(
    ['ffmpeg', '-y', '-i', copied_path, '-vn', '-ac', '1', '-ar', '16000', AUDIO_PATH],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError(f'❌ ffmpeg の変換に失敗しました:\n{result.stderr[-1500:]}')

# 4. 長さを調べて表示
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', AUDIO_PATH],
    capture_output=True, text=True,
)
try:
    AUDIO_DURATION_SEC = float(probe.stdout.strip())
except ValueError:
    raise RuntimeError(f'❌ 音声の長さが取得できませんでした:\n{probe.stderr[-500:]}')

minutes, seconds = divmod(int(AUDIO_DURATION_SEC), 60)
print(f'✅ 変換完了: {AUDIO_PATH}')
print(f'⏱️ 音声の長さ: {minutes}分{seconds}秒（{AUDIO_DURATION_SEC:.1f}秒）')
print(f'📁 結果の保存先: {EXPERIMENT_DIR}')
